# 0.1 Use Data You Already Saved

**Only for people who ran the earlier (classic Workbench) notebooks** and still have their CSV files in a bucket. This converts them into the files the new notebooks expect, so you can **skip the queries** in 1.1 and 1.2 and go straight to 2.0.

| Old file (in `data/`) | Becomes |
|---|---|
| `person_control.csv` + `person_study.csv` | `person.csv` (with a `diabetes` column) |
| `survey_control.csv` + `survey_study.csv` | `survey.csv` |
| `measurement_control.csv` + `measurement_study.csv` | `measurement.csv` |
| `condition_arranged.csv` | `survival_conditions.csv` |
| `visit_arranged.csv` | `last_visits.csv` |

Your old **`final_analysis.csv`** already works with notebooks 3.1 to 3.3. Run 2.0 anyway if you want 3.4 (it also saves `measurements_wide.csv`).

> **Note:** these files come from the older dataset (**CDR v8**) and the earlier cohort definition. Fine for practising the analyses; results won't match a fresh CDR v9 run.

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data or counts of 1 to 20. Clear outputs before you share or commit this notebook.

## Step 1: Point `aou_setup.R` at the bucket with your old files

The old notebooks saved to the `data/` folder of the workspace bucket. In a workspace **migrated** from the classic Workbench, that's the bucket named like `rw-migration-aou-rw-…`. Set `BUCKET_NAME` in `aou_setup.R` to that bucket's folder name, then:

In [ ]:
source("aou_setup.R")  # settings, packages, and helper functions

In [ ]:
list.files(DATA_DIR)

## Step 2: Convert

`load_old()` reads both cohort files of a table and adds the `diabetes` flag: *No* for the control file, *Yes* for the study file.

In [ ]:
load_old <- function(table) {
  bind_rows(
    load_data(str_glue("{table}_control.csv")) %>% mutate(diabetes = "No"),
    load_data(str_glue("{table}_study.csv"))   %>% mutate(diabetes = "Yes")
  )
}

person <- load_old("person") %>%
  transmute(person_id, diabetes, date_of_birth = as.Date(date_of_birth), gender, race, ethnicity)

survey <- load_old("survey") %>%
  select(person_id, question, answer)

measurement <- load_old("measurement") %>%
  select(person_id, measurement_concept_id, standard_concept_name, measurement_datetime, value_as_number)

person %>% count(diabetes)

**Survival files.** The old condition file has the condition **name** but no diabetes/CKD flags, so we rebuild them from the name. (The new 1.2 query uses concept codes instead, so it's not identical.) The old visit file has every visit, so we keep each person's last one.

In [ ]:
survival_conditions <- load_data("condition_arranged.csv") %>%
  transmute(
    person_id,
    condition_start_date = as.Date(condition_start_datetime),
    condition_name       = standard_concept_name,
    is_diabetes = str_detect(str_to_lower(condition_name), "diabet"),
    is_ckd      = str_detect(str_to_lower(condition_name), "kidney|renal")
  )

last_visits <- load_data("visit_arranged.csv") %>%
  rename_with(str_to_lower) %>%                     # PERSON_ID -> person_id
  group_by(person_id) %>%
  summarise(last_visit_date = max(as.Date(visit_end_datetime), na.rm = TRUE))

survival_conditions %>% count(is_diabetes, is_ckd)

## Step 3: Save in the new format

In [ ]:
save_data(person,              "person.csv")
save_data(survey,              "survey.csv")
save_data(measurement,         "measurement.csv")
save_data(survival_conditions, "survival_conditions.csv")
save_data(last_visits,         "last_visits.csv")

**Next:** skip 1.1 and 1.2 and open **2.0 clean_and_join**.